# Logistic Regression

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from time import time


In [2]:
df = pd.read_csv("datasets/airlines_delay.csv", sep=",")
AirlineUnique = df.Airline.unique()
AirportFromUnique = df.AirportFrom.unique()
AirportToUnique = df.AirportTo.unique()

Airlinelst = list(range(len(AirlineUnique)))
df['NumAirline'] = df['Airline']
df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)
AirportFromlst = list(range(len(AirportFromUnique)))
df['NumAirportFrom'] = df['AirportFrom']
df['NumAirportFrom'].replace(AirportFromUnique, AirportFromlst, inplace=True)
AirportTolst = list(range(len(AirportToUnique)))
df['NumAirportTo'] = df['AirportTo']
df['NumAirportTo'].replace(AirportToUnique, AirportTolst, inplace=True)

df = df.sample(n=10000)
X = df[["Length","NumAirline","NumAirportFrom","NumAirportTo","DayOfWeek"]]
y = df['Class']


/var/folders/61/4f_9vd3x7c9_5dsr15qd4fww0000gn/T/ipykernel_37598/2626831645.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)
/var/folders/61/4f_9vd3x7c9_5dsr15qd4fww0000gn/T/ipykernel_37598/2626831645.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_d

In [3]:
X_simulated_small, y_simulated_small = make_classification(n_samples=300, n_features=6, n_classes=2, random_state=1)
X_simulated_large, y_simulated_large = make_classification(n_samples=15000, n_features=6, n_classes=2, random_state=1)

In [4]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
X_std = X_std
y = y.values


In [5]:
def test_accuracy(true_values, predicted_values):
    correct = 0
    for a,b in zip(true_values, predicted_values):
        if a==b:
            correct +=1
    return correct/len(true_values)


## Gradient Ascent Implementation

In [6]:
class logregwMLE:
    def __init__(self, lr=0.01, numiterations=100):
        self.lr = lr
        self.numiterations = numiterations
        self.lls = []
        self.eps = 1e-10
    def sig(self, z):
        return 1/(1+np.exp(-z))
    def logL(self, ycorrect, predY):
        predY = np.maximum(np.full(predY.shape, self.eps), np.minimum(np.full(predY.shape,1-self.eps), predY))
        ll = ycorrect*np.log(predY)+(1-ycorrect)*np.log(1-predY)
        return ll.mean()
    def fit(self, X, y):
        self.wghs = np.zeros(X.shape[1])
        for _ in range(self.numiterations):
            z = X @ self.wghs
            predY = self.sig(z)
            grad = np.mean((y-predY)*X.T, axis=1)
            self.wghs += self.lr*grad
            ll = self.logL(y,predY)
            self.lls.append(ll)
    def predprob(self, X):
        z = X @ self.wghs
        return self.sig(z)
    def predict(self, X, threshold=0.5):
        return np.array([1 if p>threshold else 0 for p in self.predprob(X)])


In [7]:
# Simulated Study Gradient Ascent (small data)
start = time()
model = logregwMLE()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_simulated_small, y_simulated_small, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc.append(test_accuracy(y_test, pred))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression with Gradient Ascent:', time()-start)


Prediction accuracy of model: 0.9133333333333333
Training time for Logistic Regression with Gradient Ascent: 0.036786794662475586


In [8]:
# Simulated Study Gradient Ascent (large data)
start = time()
model = logregwMLE()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_simulated_large, y_simulated_large, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc.append(test_accuracy(y_test, pred))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression with Gradient Ascent:', time()-start)


Prediction accuracy of model: 0.8605333333333333
Training time for Logistic Regression with Gradient Ascent: 0.6177058219909668


In [9]:
# Real data study
start = time()
model = logregwMLE()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc.append(test_accuracy(y_test, pred))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression with Gradient Ascent:', time()-start)


Prediction accuracy of model: 0.5525
Training time for Logistic Regression with Gradient Ascent: 0.3744809627532959


## Gradient Descent Implementation

In [10]:
import math

def sig(X, weight):
    z = X @ weight
    return 1/(1+np.exp(-z))

def graddescent(X, g, y):
    return (X.T @ (g - y)) / y.shape[0]

def new_weightloss(weight, lr, grad):
    return weight - lr * grad

def generateX(X):
    return np.c_[np.ones((len(X),1)), X]

def get_initial_vec(X):
    return np.random.randn(X.shape[1]+1, 1)

def sigmoid_function(X):
    return 1/(1+math.e**(-X))

def Logistic_Fit(X,y,learningrate,iterations):
    y_new = y.reshape(len(y),1)
    vectorX = generateX(X)
    theta = get_initial_vec(X)
    m = len(X)
    for _ in range(iterations):
        gradients = 2/m * vectorX.T.dot(sigmoid_function(vectorX.dot(theta)) - y_new)
        theta = theta - learningrate * gradients
    return theta

def column(matrix, i):
    return [row[i] for row in matrix]

def accuracy_metric(X,y,learningrate,iteration,X_test,y_test):
    fit_model = Logistic_Fit(X,y,learningrate,iteration)
    line = fit_model[0]
    for i in range(1,len(fit_model)):
        line = line + fit_model[i]*column(X_test,i-1)
    logistic_function = sigmoid_function(line)
    logistic_function = np.where(logistic_function>=0.5, 1, 0)
    count = np.sum(logistic_function.reshape(-1,1)==y_test.reshape(-1,1))
    return count/len(y_test)


In [11]:
# Gradient Descent - small simulated data
start = time()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_simulated_small, y_simulated_small, test_size=0.2, random_state=i)
    acc.append(accuracy_metric(X_train,y_train,0.1,100,X_test,y_test))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression using Gradient Descent:', time()-start)


Prediction accuracy of model: 0.9100000000000001
Training time for Logistic Regression using Gradient Descent: 0.016477108001708984


In [12]:
# Gradient Descent - large simulated data
start = time()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_simulated_large, y_simulated_large, test_size=0.2, random_state=i)
    acc.append(accuracy_metric(X_train,y_train,0.1,100,X_test,y_test))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression using Gradient Descent:', time()-start)


Prediction accuracy of model: 0.8981999999999999
Training time for Logistic Regression using Gradient Descent: 0.3194310665130615


In [13]:
# Gradient Descent - real data
start = time()
acc = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.2, random_state=i)
    acc.append(accuracy_metric(X_train,y_train,0.1,100,X_test,y_test))
print('Prediction accuracy of model:', sum(acc)/len(acc))
print('Training time for Logistic Regression using Gradient Descent:', time()-start)


Prediction accuracy of model: 0.5516
Training time for Logistic Regression using Gradient Descent: 0.2049400806427002
